# ต่อยอดสู่ AI: สอนคอมพิวเตอร์ให้ "จำ" แทนการเขียนกฎ
ที่ผ่านมาเรา**เขียนกฎ**เอง (สีแดง H 0–10, พื้นที่ > 1000, ต่างจากภาพอ้างอิง > 20) กฎทำงานดีเมื่อชิ้นงานหน้าตาคงที่ แต่พอ**ชิ้นงานขยับ แสงเปลี่ยน หรือตำหนิมีหลายแบบ** กฎจะยาวขึ้นเรื่อยๆ และพังง่าย

**AI (Machine Learning)** กลับทาง: เราให้**ตัวอย่างภาพพร้อมเฉลย**จำนวนมาก แล้วให้คอมพิวเตอร์หากฎเอง

notebook นี้ทำครบวงจรแบบย่อ ใน 2 นาที บนเครื่องธรรมดา (ไม่ต้องมีการ์ดจอ):
1. สร้างชุดข้อมูล 3 ประเภท: ถาดดี / ชิ้นหาย / มีสิ่งแปลกปลอม  ชิ้นงาน**ขยับและแสงเปลี่ยน**ทุกภาพ
2. แปลงภาพเป็น "ลักษณะเด่น" (Features) แล้วสอนตัวจำแนก (Classifier)
3. วัดความแม่น และดูว่าผิดตรงไหน
4. เทียบกับวิธีเขียนกฎ (เทียบภาพอ้างอิง) ว่าใครรับมือการขยับได้
5. ใช้กับภาพจากกล้อง และทางไปต่อสู่ Deep Learning

ต้องมี: `pip install scikit-learn` (อยู่ใน requirements.txt แล้ว)

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath("../scripts"))
import cv2
import numpy as np
from vision_camera import show, img_path, grab_one
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
print("พร้อม")

## 1. สร้างชุดข้อมูล (Dataset)
ในโรงงานจริงขั้นนี้คือ**ถ่ายภาพจริง**หลายร้อยภาพแล้วให้คนติดป้าย (Label) ว่าเป็นอะไร เป็นงานที่ใช้เวลามากที่สุดของโครงการ AI
ที่นี่สร้างภาพจำลองแทน โดยจงใจให้**ชิ้นงานขยับ ±10 พิกเซล แสงเปลี่ยน 60–130% และมีจุดรบกวน** เหมือนหน้างานที่จิ๊กหลวม

In [ ]:
CLASSES = ["ok", "missing", "foreign"]

def make_tray(kind, rng):
    img = np.full((240, 320, 3), (70, 72, 75), np.uint8)
    dx, dy = rng.integers(-10, 11, 2)                              # ถาดขยับ (จิ๊กหลวม)
    centers = [(60 + dx, 60 + dy), (160 + dx, 60 + dy), (260 + dx, 60 + dy),
               (60 + dx, 170 + dy), (160 + dx, 170 + dy), (260 + dx, 170 + dy)]
    skip = rng.integers(0, 6) if kind == "missing" else -1
    for i, (x, y) in enumerate(centers):
        if i == skip:
            continue
        cv2.circle(img, (int(x), int(y)), 26, (190, 190, 195), -1)
        cv2.circle(img, (int(x), int(y)), 8, (80, 82, 85), -1)
    if kind == "foreign":                                           # สิ่งแปลกปลอม สุ่มชนิด ตำแหน่ง ขนาด
        fx, fy = int(rng.integers(30, 290)), int(rng.integers(30, 210))
        f = rng.integers(0, 3)
        if f == 0:                                                   # สกรู
            cv2.rectangle(img, (fx, fy), (fx + int(rng.integers(40, 70)), fy + 12), (30, 30, 35), -1)
        elif f == 1:                                                 # เศษพลาสติกสี
            cv2.circle(img, (fx, fy), int(rng.integers(14, 22)), (60, 200, 230), -1)
        else:                                                        # เส้นด้าย
            pts = np.array([[fx - 30, fy], [fx, fy - 12], [fx + 30, fy + 10], [fx + 60, fy - 5]], np.int32)
            cv2.polylines(img, [pts], False, (20, 20, 20), 3)
    light = rng.uniform(0.6, 1.3)                                   # แสงเปลี่ยน
    img = np.clip(img.astype(np.float32) * light + rng.normal(0, 3, img.shape), 0, 255).astype(np.uint8)
    return img

rng = np.random.default_rng(0)
X_img, y = [], []
for ci, kind in enumerate(CLASSES):
    for _ in range(100):
        X_img.append(make_tray(kind, rng)); y.append(ci)
y = np.array(y)
print("ภาพทั้งหมด", len(X_img), " ต่อประเภท 100 ภาพ")
show(X_img[0], X_img[80], X_img[160], titles=CLASSES, size=4)

## 2. แปลงภาพเป็น Features แล้วสอนโมเดล
คอมพิวเตอร์เรียนจากตัวเลข จึงแปลงภาพเป็นเวกเตอร์ตัวเลขที่สรุป "ลักษณะเด่น":
- **HOG** (Histogram of Oriented Gradients): สรุปว่าขอบในภาพหันทางไหนบ้าง จับรูปทรงได้โดยไม่สนใจตำแหน่งเป๊ะๆ
- **ภาพย่อ** 32 × 24 พิกเซล หารด้วยความสว่างเฉลี่ย (ตัดผลของแสง)
- **Histogram สี**: สัดส่วนสีในภาพ จับสิ่งแปลกปลอมสีต่างได้

จากนั้นใช้ **SVM** (Support Vector Machine) ซึ่งเป็น Machine Learning แบบคลาสสิก สอนได้ในไม่กี่วินาที

In [ ]:
from vision_camera import make_hog
hog = make_hog()   # OpenCV 4 ใช้ HOG ของ OpenCV  OpenCV 5 ใช้ตัวที่เขียนเองแทน

def features(img):
    small = cv2.resize(img, (64, 48))
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    f_shape = hog.compute(gray).flatten()                          # รูปทรง
    thumb = cv2.resize(gray, (32, 24)).astype(np.float32)
    f_thumb = (thumb / (thumb.mean() + 1e-6)).flatten()            # ภาพย่อ ตัดผลของแสง
    hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
    f_color = cv2.calcHist([hsv], [0, 1], None, [8, 4], [0, 180, 0, 256]).flatten()
    f_color = 10 * f_color / (f_color.sum() + 1e-6)                # สี (สัดส่วน ไม่ขึ้นกับความสว่าง)
    return np.concatenate([f_shape, f_thumb, f_color])

X = np.array([features(im) for im in X_img])
print("Features ต่อภาพ:", X.shape[1], "ตัวเลข")

# แบ่งข้อมูล: 70% ใช้สอน  30% เก็บไว้สอบ (โมเดลไม่เคยเห็น)
X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    X, y, X_img, test_size=0.3, random_state=0, stratify=y)

t0 = time.perf_counter()
model = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10)).fit(X_train, y_train)
print(f"สอนโมเดลใช้เวลา {time.perf_counter() - t0:.2f} วินาที  (ภาพสอน {len(y_train)} ภาพ)")

## 3. วัดความแม่นกับภาพที่โมเดลไม่เคยเห็น
**Confusion matrix:** แถว = เฉลยจริง  คอลัมน์ = ที่โมเดลทาย  ตัวเลขนอกแนวทแยง = ทายผิด

In [ ]:
pred = model.predict(X_test)
print(f"ความแม่น (Accuracy): {accuracy_score(y_test, pred) * 100:.1f}%  จากภาพสอบ {len(y_test)} ภาพ")
cm = confusion_matrix(y_test, pred)
print("           ทาย:", CLASSES)
for name, row in zip(CLASSES, cm):
    print(f"จริง {name:8s}", row)

wrong = [i for i in range(len(y_test)) if pred[i] != y_test[i]]
if wrong:
    show(*[img_test[i] for i in wrong[:3]],
         titles=[f"จริง {CLASSES[y_test[i]]} ทาย {CLASSES[pred[i]]}" for i in wrong[:3]], size=4)
else:
    print("ไม่มีภาพที่ทายผิด")

## 4. เทียบกับวิธีเขียนกฎ: ใครรับมือ "ชิ้นงานขยับ" ได้
ลองใช้วิธี**เทียบภาพอ้างอิง**จาก notebook 06 กับชุดข้อมูลเดียวกัน (ที่ชิ้นงานขยับและแสงเปลี่ยน)

In [ ]:
ref = make_tray("ok", np.random.default_rng(999))           # ภาพอ้างอิง 1 ภาพ

def rule_based(img, ref, th=35, min_area=120):
    g1 = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    g0 = cv2.GaussianBlur(cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, m = cv2.threshold(cv2.absdiff(g1, g0), th, 255, cv2.THRESH_BINARY)
    cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return 0 if not any(cv2.contourArea(c) > min_area for c in cs) else 1   # 0 = บอกว่า ok, 1 = บอกว่า NG

rule_pred = np.array([rule_based(im, ref) for im in img_test])
truth_ng = (y_test != 0).astype(int)
rule_acc = (rule_pred == truth_ng).mean() * 100
ai_acc = ((pred != 0).astype(int) == truth_ng).mean() * 100
print(f"ตัดสิน OK/NG ถูกต้อง   เขียนกฎ (เทียบภาพอ้างอิง): {rule_acc:.0f}%    AI (SVM): {ai_acc:.0f}%")
print("ชุดข้อมูลนี้ชิ้นงานขยับทุกภาพ กฎเทียบภาพอ้างอิงจึงเห็น 'ทุกอย่างต่าง' และบอก NG แม้ถาดดี")

**บทเรียน:** กฎเทียบภาพอ้างอิงแม่นมากเมื่อมีจิ๊กยึด (notebook 06) แต่พังทันทีเมื่อชิ้นงานขยับ ส่วน AI เรียนรู้ความหลากหลายจากตัวอย่าง จึงทนต่อการขยับและแสงได้  **แลกกับ**การต้องมีภาพตัวอย่างพร้อมเฉลยจำนวนมาก และอธิบายเหตุผลได้ยากกว่า

ในโรงงานจริงมักใช้**ทั้งสองอย่าง**: จิ๊ก + กฎง่ายๆ สำหรับสิ่งที่นิยามชัด และ AI สำหรับตำหนิที่หลากหลาย

## 5. ใช้โมเดลกับภาพจากกล้อง
ขั้นตอนเดียวกับทุก workshop: ถ่ายภาพ → แปลงเป็น Features → ให้โมเดลทาย
(ภาพจากกล้องจำลองเป็นภาพรูปทรงหลากสี ไม่ใช่ถาด ผลที่ทายจึงไม่มีความหมาย แค่แสดงให้เห็นวิธีต่อ  กับกล้องจริงต้องสร้างชุดข้อมูลจากภาพจริงก่อน)

In [ ]:
frame = grab_one("emu", image="shapes.png")
frame = cv2.resize(frame, (320, 240))
k = model.predict([features(frame)])[0]
print("โมเดลทายว่า:", CLASSES[k])
show(frame, titles=["ภาพจากกล้อง -> " + CLASSES[k]], size=4)

## 6. ทางไปต่อ: Deep Learning และเครื่องมือที่ใช้จริง
SVM + HOG พอสำหรับงานง่ายๆ แต่ตำหนิจริง (รอยขีด รอยบุบ สีเพี้ยนเล็กน้อย) ต้องการ **Deep Learning** ซึ่งเรียน Features เองจากภาพ

| เครื่องมือ | เหมาะกับ | ต้องเขียนโค้ดไหม |
|---|---|---|
| **Teachable Machine** (Google, ในเบราว์เซอร์) | ลองแนวคิด Image Classification ด้วย webcam ใน 5 นาที | ไม่ต้อง |
| **Roboflow** | ติดป้ายภาพ (Label) จัดการชุดข้อมูล สอน Object Detection | น้อย |
| **Ultralytics YOLO** (Python) | Classification / Detection / Segmentation ใช้ในงานจริงได้ | เขียนไม่กี่บรรทัด |
| **Basler pylon AI / vTools** | ใช้ AI กับกล้อง Basler โดยตรง | ตั้งค่าในโปรแกรม |

ตัวอย่างสอน YOLO จำแนกภาพ (ต้อง `pip install ultralytics` ประมาณ 1 GB และใช้เวลาสอนหลายนาทีถ้าไม่มีการ์ดจอ):
```python
from ultralytics import YOLO
model = YOLO("yolov8n-cls.pt")                 # โมเดลที่สอนมาแล้ว นำมาสอนต่อ (Transfer learning)
model.train(data="dataset/", epochs=20)        # dataset/train/ok, dataset/train/ng, dataset/val/...
result = model("tray.png")                     # ทาย
```

**สิ่งที่ต้องเตรียมก่อนทำ AI ในโรงงาน**
1. **ภาพจริงจากกล้องจริงที่ตำแหน่งจริง** อย่างน้อยหลักร้อยภาพต่อประเภท ครอบคลุมทุกกะ ทุกสภาพแสง
2. **เฉลยที่ถูกต้อง** ให้ QC ที่เชี่ยวชาญเป็นคนติดป้าย
3. **ชุดสอบที่แยกไว้** ไม่เคยใช้สอน เพื่อวัดความแม่นจริง
4. **เกณฑ์ยอมรับ** เช่น หลุดรอดไม่เกิน 0.1% (เชื่อมกับเรื่องต้นทุนใน notebook 05)
5. **ทางถอย** ถ้าโมเดลไม่มั่นใจ ให้ส่งคนตรวจ ไม่ใช่ปล่อยผ่าน